In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import plotly.express as px 

In [ ]:
nltk.download('vader_lexicon')

In [ ]:
# Step 1: Load the Dataset
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')

In [ ]:
# Step 2: Data Cleaning
apps_df = apps_df.dropna(subset=['Rating'])
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)
apps_df.drop_duplicates(inplace=True)
apps_df = apps_df[apps_df['Rating'] <= 5]
reviews_df.dropna(subset=['Translated_Review'], inplace=True)

In [ ]:
# Merge datasets on 'App' and handle non-matching apps
merged_df = pd.merge(apps_df, reviews_df, on='App', how='inner')

In [ ]:
# Step 3: Data Transformation
apps_df['Reviews'] = apps_df['Reviews'].astype(int)
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '').str.replace('+', '').astype(int)
apps_df['Price'] = apps_df['Price'].str.replace('$', '').astype(float)

In [ ]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    else:
        return np.nan

In [ ]:
apps_df['Size'] = apps_df['Size'].apply(convert_size)

In [ ]:
# Add log_installs and log_reviews columns
apps_df['Log_Installs'] = np.log1p(apps_df['Installs'])
apps_df['Log_Reviews'] = np.log1p(apps_df['Reviews'])

In [ ]:
#task 2 add create cuntry colum 
apps_df["Country"] = "India"

In [ ]:
# Add Rating Group column
def rating_group(rating):
    if rating >= 4:
        return 'Top rated'
    elif rating >= 3:
        return 'Above average'
    elif rating >= 2:
        return 'Average'
    else:
        return 'Below average'

apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)

In [ ]:
# Add Revenue column
apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

In [ ]:
# Sentiment Analysis
sia = SentimentIntensityAnalyzer()
reviews_df['Sentiment_Score'] = reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [ ]:
#task 2 : create countrt colum 
countries = [
    "India", "United States", "United Kingdom",
     "Canada", "Australia", "Germany",
    "France", "Japan", "Brazil", "Singapore"
]

apps_df["Country"] = [
    countries[i % len(countries)] for i in range(len(apps_df))
]
apps_df[["App", "Country"]].head()


In [ ]:
# Extract year from 'Last Updated' and create 'Year' column
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df['Year'] = apps_df['Last Updated'].dt.year

In [ ]:
# Create Tkinter window
class AppDashboard(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Google Play Store Analysis Dashboard")
        self.geometry("1200x800")
        self.configure(bg='lightgray')

        # Create a main canvas and scrollbars
        main_frame = tk.Frame(self)
        main_frame.pack(fill=tk.BOTH, expand=True)

        canvas = tk.Canvas(main_frame, bg='lightgray')
        v_scrollbar = ttk.Scrollbar(main_frame, orient="vertical", command=canvas.yview)
        h_scrollbar = ttk.Scrollbar(main_frame, orient="horizontal", command=canvas.xview)
        
        v_scrollbar.pack(side="right", fill="y")
        h_scrollbar.pack(side="bottom", fill="x")
        canvas.pack(side="left", fill="both", expand=True)
        
        scrollable_frame = ttk.Frame(canvas)
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(
                scrollregion=canvas.bbox("all")
            )
        )

        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=v_scrollbar.set, xscrollcommand=h_scrollbar.set)

        # Create a grid layout for the dashboard
        for i in range(6):
            scrollable_frame.columnconfigure(i, weight=1)

        # Create and place frames for each visualization
        self.create_category_analysis(scrollable_frame, 0, 0)
        self.create_type_analysis(scrollable_frame, 0, 1)
        self.create_rating_sentiment_analysis(scrollable_frame, 0, 2)
        self.create_installation_update_analysis(scrollable_frame, 0, 3)
        self.create_additional_insights(scrollable_frame, 0, 4)
        self.create_ml_model_evaluation(scrollable_frame, 0, 5)
        #add task 1
        self.create_bubble_chart(scrollable_frame, 1, 0)
        #add task 2
        self.create_choropleth_map(scrollable_frame, 1, 1)
        #task 3
        self.create_time_series_chart(scrollable_frame, 2, 0)
        # Task 4
        self.create_stacked_area_chart(scrollable_frame, 2, 1)
        #Task 5
        self.create_grouped_bar_chart(scrollable_frame, 2, 2)
        # Task 6
        self.create_dual_axis_chart(scrollable_frame, 2, 3)
        





    def create_category_analysis(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        category_counts = apps_df['Category'].value_counts().nlargest(10)
        fig, ax = plt.subplots(figsize=(5, 3))
        sns.barplot(x=category_counts.values, y=category_counts.index, palette='Spectral', ax=ax)
        ax.set_title('Top Categories on Play Store')
        ax.set_xlabel('Number of Apps')
        ax.set_ylabel('Category')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    def create_type_analysis(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        type_counts = apps_df['Type'].value_counts()
        fig, ax = plt.subplots(figsize=(3, 2))
        ax.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999', '#66b3ff'])
        ax.set_title('Distribution of Free and Paid Apps')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    def create_rating_sentiment_analysis(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        fig, ax = plt.subplots(figsize=(5, 3))
        sns.histplot(apps_df['Rating'], bins=30, kde=True, color='purple', ax=ax)
        ax.set_title('Rating Distribution')
        ax.set_xlabel('Rating')
        ax.set_ylabel('Frequency')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

        sentiment_counts = reviews_df['Sentiment_Score'].apply(lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral')).value_counts()
        fig, ax = plt.subplots(figsize=(3, 2))
        ax.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140, colors=['#ffcc99', '#99ff99', '#66ffcc'])
        ax.set_title('Sentiment Distribution')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    def create_installation_update_analysis(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        installs_by_category = apps_df.groupby('Category')['Installs'].sum().nlargest(10)
        fig, ax = plt.subplots(figsize=(5, 3))
        sns.barplot(x=installs_by_category.values, y=installs_by_category.index, palette='coolwarm', ax=ax)
        ax.set_title('Installations by Category')
        ax.set_xlabel('Number of Installations')
        ax.set_ylabel('Category')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

        updates_over_year = apps_df['Year'].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(5, 3))
        sns.lineplot(x=updates_over_year.index, y=updates_over_year.values, marker='o', ax=ax)
        ax.set_title('Distribution of App Updates Over the Year')
        ax.set_xlabel('Year')
        ax.set_ylabel('Number of Updates')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    def create_additional_insights(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        revenue_by_category = apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
        fig, ax = plt.subplots(figsize=(5, 3))
        sns.barplot(x=revenue_by_category.values, y=revenue_by_category.index, palette='viridis', ax=ax)
        ax.set_title('Revenue by Category')
        ax.set_xlabel('Revenue')
        ax.set_ylabel('Category')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

        genres_counts = apps_df['Genres'].str.split(';').explode().value_counts().nlargest(10)
        fig, ax = plt.subplots(figsize=(5, 3))
        sns.barplot(x=genres_counts.values, y=genres_counts.index, palette='magma', ax=ax)
        ax.set_title('Top Genres on Play Store')
        ax.set_xlabel('Number of Apps')
        ax.set_ylabel('Genres')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

        fig, ax = plt.subplots(figsize=(5, 3))
        sns.boxplot(data=apps_df, x='Year', y='Rating', palette='cool', ax=ax)
        ax.set_title('Effect of Last Update on Rating')
        ax.set_xlabel('Year')
        ax.set_ylabel('Rating')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

        fig, ax = plt.subplots(figsize=(5, 3))
        sns.boxplot(data=apps_df, x='Type', y='Rating', palette='Set2', ax=ax)
        ax.set_title('Ratings for Paid vs Free Apps')
        ax.set_xlabel('Type')
        ax.set_ylabel('Rating')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    def create_ml_model_evaluation(self, parent, row, column):
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)
        frame.columnconfigure(0, weight=1)

        X = apps_df[['Log_Reviews', 'Log_Installs', 'Price']]
        y = apps_df['Rating']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        fig, ax = plt.subplots(figsize=(5, 3))
        ax.scatter(y_test, y_pred, alpha=0.3)
        ax.plot([0, 5], [0, 5], 'r--')
        ax.set_title(f'ML Model Evaluation (MSE: {mse:.2f}, R2: {r2:.2f})')
        ax.set_xlabel('True Rating')
        ax.set_ylabel('Predicted Rating')
        fig.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(expand=True, fill="both")

    #add task chart 1
    def create_bubble_chart(self, parent, row, column):
        from datetime import datetime
        import pytz
        import plotly.express as px

        ist = pytz.timezone("Asia/Kolkata")
        current_time = datetime.now(ist).hour

         # Show chart only between 5 PM and 7 PM
        if current_time < 17 or current_time >= 19:
          return
        
        frame = ttk.Frame(parent, padding="5")
        frame.grid(row=row, column=column, sticky="nsew", pady=5)

        # Merge apps and reviews
        subjectivity = reviews_df.groupby("App")["Sentiment_Subjectivity"].mean().reset_index()

        bubble_df = apps_df.merge(subjectivity, on="App", how="left")

        categories = [
            "GAME", 
            "BEAUTY", 
            "BUSINESS",
            "COMICS",
            "COMMUNICATION",
            "DATING",
            "ENTERTAINMENT",
            "SOCIAL", 
            "EVENTS"
        ]

        bubble_df = bubble_df[
            (bubble_df["Rating"] > 3.5) &
            (bubble_df["Reviews"] > 500) &
            (bubble_df["Installs"] > 50000) &
            (bubble_df["Sentiment_Subjectivity"] > 0.5) &
            (bubble_df["Category"].isin(categories)) &
             (~bubble_df["App"].str.contains("S", case=False, na=False))
        ].copy()

         # Translate Categories
        bubble_df["Category"] = bubble_df["Category"].replace({
            "BEAUTY": "सौंदर्य",
            "BUSINESS": "வணிகம்",
            "DATING": "Verabredung"
         })
        
        # Bubble Color
        bubble_df["Color"] = bubble_df["Category"].apply(
            lambda x: "pink" if x == "GAME" else "skyblue"
        )

        fig, ax = plt.subplots(figsize=(7,5))

        colors = bubble_df["Category"].apply(
            lambda x: "pink" if x == "GAME" else "skyblue"
        )

        ax.scatter(
            bubble_df["Size"],
            bubble_df["Rating"],
            s=bubble_df["Installs"] / 10000,
            c=colors,
            alpha=0.6
        )

        ax.set_title("App Size vs Rating")
        ax.set_xlabel("App Size (MB)")
        ax.set_ylabel("Average Rating")

        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill="both", expand=True)
    
    #chart task 2
    def create_choropleth_map(self, parent, row, column):
        from datetime import datetime
        import pytz
        #time condition
        ist = pytz.timezone("Asia/Kolkata")
        current_hour = datetime.now(ist).hour

        if current_hour < 18 or current_hour >= 20:
            return
        
        frame = ttk.Frame(parent, padding=5)
        frame.grid(row=row, column=column, sticky="nsew")
        
        #Category Filter
        df = apps_df.copy()
        df = df[
            ~df["Category"].str.startswith(("A", "C", "G", "S"))
        ].copy()
        print(apps_df.columns)
        
        #Top 5 Categories
        top5 = df.groupby("Category")["Installs"].sum().nlargest(5)
        df = df[df["Category"].isin(top5.index)]
        
        #Highlight Installs > 1 Million
        df["Highlight"] = df["Installs"].apply(
            lambda x: "More than 1M" if x > 1000000 else "Less than 1M"
        )
        
        #Choropleth Map
        fig = px.choropleth(
            df,
            locations="Country",
            locationmode="country names",
            color="Highlight",
            hover_name="Category",
            title="Global Installs by Category"
        )

        fig.show()

    #task 3: create_time_series_chart()
    def create_time_series_chart(self, frame, row, column):
        df = apps_df.copy()

        from datetime import datetime
        import pytz
        # Show only between 6 PM and 9 PM
        ist = pytz.timezone("Asia/Kolkata")
        now = datetime.now(ist)
        if not (18 <= now.hour < 21):
           return

        df = apps_df.copy()
        print("Original:", df.shape)
        
        # Reviews > 500
        df = df[df["Reviews"] > 500]
        print("Reviews:", df.shape)

        # App name should not start with X,Y,Z
        df = df[~df["App"].str.startswith(("X","Y","Z"), na=False)]
        print("Startswith XYZ:", df.shape)


        # App name should not contain S
        df = df[~df["App"].str.contains("S", case=False, na=False)]
        print("Contains S:", df.shape)

        # Category starts with B,C,E
        df = df[df["Category"].str.startswith(("B","C","E"))]
        print("Category BCE:", df.shape)

        # Translate categories
        df["Category"] = df["Category"].replace({
            "Beauty":"सौंदर्य",
            "Business":"வணிகம்",
            "Dating":"Partnersuche"
        })

        df["Installs"] = (
            df["Installs"]
            .astype(str)
            .str.replace(",", "")
            .str.replace("+", "", regex=False)
            .astype(int)
        )
         # Month           
        df["Month"] = pd.to_datetime(df["Last Updated"]).dt.to_period("M").astype(str)

        # Total installs
        trend = df.groupby(["Month","Category"])["Installs"].sum().reset_index()
        print(trend.head())
        

        fig = px.line(
            trend,
            x="Month",
            y="Installs",
            color="Category",
            markers=True,
            title="Monthly Installs Trend by Category"
        )

        # Month-over-Month Growth
        trend["Growth"] = trend.groupby("Category")["Installs"].pct_change() * 100

        # Highlight where Growth > 20%
        for category in trend["Category"].unique():
            temp = trend[trend["Category"] == category]
            high = temp[temp["Growth"] > 20]
            if not high.empty:

                fig.add_scatter(
                    x=high["Month"],
                    y=high["Installs"],
                    mode="lines",
                    fill="tozeroy",
                    line=dict(width=2),
                    opacity=0.3,
                    name=f"{category} >20% Growth"
                    
                )

        fig.update_layout(
            title="Monthly Installs Trend by Category",
            xaxis_title="Month",
            yaxis_title="Total Installs",
            hovermode="x unified"
        )
        
        fig.show()

    #task 4: create_time_series_chart()
    def create_stacked_area_chart(self, frame, row, column):
        from datetime import datetime
        import pytz
        import plotly.express as px

        # Show only between 4 PM and 6 PM IST
        ist = pytz.timezone("Asia/Kolkata")
        now = datetime.now(ist)
        if not (16 <= now.hour < 18):
            return

        df = apps_df.copy()

        # Rating >= 4.2
        df = df[df["Rating"] >= 4.2]

        # Reviews > 1000
        df = df[df["Reviews"] > 1000]

        # App name should not contain numbers
        df = df[~df["App"].str.contains(r"\d", regex=True, na=False)]

         # Category starts with T or P
        df = df[df["Category"].str.startswith(("T", "P"))]

        # Size between 20 and 80 MB
        df["Size"] = (
            df["Size"]
            .astype(str)
            .str.replace("M", "", regex=False)
            .str.replace("k", "", regex=False)
        )

        df["Size"] = pd.to_numeric(df["Size"], errors="coerce")
        df = df[(df["Size"] >= 20) & (df["Size"] <= 80)]

        # Convert Installs
        df["Installs"] = (
            df["Installs"]
            .astype(str)
            .str.replace(",", "")
            .str.replace("+", "", regex=False)
            .astype(int)
        )

        # Date
        df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
        df = df.dropna(subset=["Last Updated"])

        df["Month"] = df["Last Updated"].dt.to_period("M").astype(str)

        # Translate Categories
        df["Category"] = df["Category"].replace({
            "TRAVEL_AND_LOCAL": "Voyage et Local",
            "PRODUCTIVITY": "Productividad",
            "PHOTOGRAPHY": "写真"
        })

        trend = df.groupby(["Month", "Category"])["Installs"].sum().reset_index()

        # Growth
        trend["Growth"] = trend.groupby("Category")["Installs"].pct_change() * 100

        fig = px.area(
            trend,
            x="Month",
            y="Installs",
            color="Category",
            title="Stacked Area Chart - Installs by Category"
        )

        # Highlight growth >25%
        for cat in trend["Category"].unique():
            temp = trend[trend["Category"] == cat]
            high = temp[temp["Growth"] > 25]

            if not high.empty:
                fig.add_scatter(
                    x=high["Month"],
                    y=high["Installs"],
                    mode="markers",
                    marker=dict(size=10),
                    name=f"{cat} >25% Growth"
                )

            fig.update_layout(
                xaxis_title="Month",
                yaxis_title="Cumulative Installs",
                hovermode="x unified"
            )

        fig.show()

    #task 5: Grouped Bar Chart 
    def create_grouped_bar_chart(self, frame, row, column):   

        from datetime import datetime
        import pytz
        import plotly.express as px

        # Show only between 3 PM and 5 PM IST
        ist = pytz.timezone("Asia/Kolkata")
        now = datetime.now(ist)
        if not (15 <= now.hour < 17):
            return

        df = apps_df.copy()

        # Rating >= 4.0
        df = df[df["Rating"] >= 4.0]

        # Convert Size
        df["Size"] = (
            df["Size"]
            .astype(str)
            .str.replace("M", "", regex=False)
            .str.replace("k", "", regex=False)
        )

        df["Size"] = pd.to_numeric(df["Size"], errors="coerce")

        # Size >= 10 MB
        df = df[df["Size"] >= 10]

        # Date
        df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
        df = df.dropna(subset=["Last Updated"])

        # January only
        df = df[df["Last Updated"].dt.month == 1]

        # Convert Installs
        df["Installs"] = (
            df["Installs"]
            .astype(str)
            .str.replace(",", "")
            .str.replace("+", "", regex=False)
            .astype(int)
        )

        # Top 10 categories by installs
        top_cat = (
            df.groupby("Category")["Installs"]
            .sum()
            .nlargest(10)
            .index
        )

        df = df[df["Category"].isin(top_cat)]

        result = (
            df.groupby("Category")
            .agg(
                Average_Rating=("Rating", "mean"),
                Total_Reviews=("Reviews", "sum")
            )
            .reset_index()
        )

        # Convert to long format
        plot_df = result.melt(
            id_vars="Category",
            value_vars=["Average_Rating", "Total_Reviews"],
            var_name="Metric",
            value_name="Value"
        )

        fig = px.bar(
            plot_df,
            x="Category",
            y="Value",
            color="Metric",
            barmode="group",
            title="Average Rating vs Total Reviews (Top 10 Categories)"
        )

        fig.update_layout(
            xaxis_title="Category",
        yaxis_title="Value"
        )

        fig.show()

    #Task 6:Dual Axis Chart
    def create_dual_axis_chart(self, frame, row, column):
        from datetime import datetime
        import pytz
        import plotly.graph_objects as go

        # Show only between 1 PM and 2 PM IST
        # ist = pytz.timezone("Asia/Kolkata")
        # now = datetime.now(ist)
        # if not (13 <= now.hour < 14):
        #     return

        df = apps_df.copy()

        # Convert Installs
        df["Installs"] = (
            df["Installs"]
            .astype(str)
            .str.replace(",", "")
            .str.replace("+", "", regex=False)
        )

        df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

        # Size
        df["Size"] = (
            df["Size"]
            .astype(str)
            .str.replace("M", "", regex=False)
            .str.replace("k", "", regex=False)
        )

        df["Size"] = pd.to_numeric(df["Size"], errors="coerce")

        # Android Version
        df["Android Ver"] = (
            df["Android Ver"]
            .astype(str)
            .str.extract(r'(\d+\.?\d*)')[0]
        )

        df["Android Ver"] = pd.to_numeric(df["Android Ver"], errors="coerce")

        # Filters
        df = df[df["Installs"] >= 10000]
        df = df[df["Revenue"] >= 10000]
        df = df[df["Android Ver"] > 4.0]
        df = df[df["Size"] > 15]
        df = df[df["Content Rating"] == "Everyone"]

        # App name <= 30 characters
        df = df[df["App"].str.len() <= 30]

        # Top 3 Categories by Installs
        top3 = (
            df.groupby("Category")["Installs"]
            .sum()
            .nlargest(3)
            .index
        )

        df = df[df["Category"].isin(top3)]

        result = (
            df.groupby("Type")
            .agg(
                Avg_Installs=("Installs", "mean"),
                Avg_Revenue=("Revenue", "mean")
            )
            .reset_index()
        )

        fig = go.Figure()

        # Average Installs
        fig.add_bar(
            x=result["Type"],
            y=result["Avg_Installs"],
            name="Average Installs",
            yaxis="y1"
        )

        # Revenue
        fig.add_scatter(
            x=result["Type"],
            y=result["Avg_Revenue"],
            mode="lines+markers",
            name="Average Revenue",
            yaxis="y2"
        )

        fig.update_layout(
            title="Average Installs vs Revenue (Free vs Paid Apps)",
            xaxis=dict(title="App Type"),
            yaxis=dict(title="Average Installs"),
            yaxis2=dict(
                title="Average Revenue",
                overlaying="y",
                side="right"
            ),
            hovermode="x unified"
        )

        fig.show()

if __name__ == "__main__":
    app = AppDashboard()
    app.mainloop()